# ECG-FM encoder on a full 5000-sample record

**Question.** The pipeline in `src/data.py` feeds the encoder 2500-sample (5 s @ 500 Hz)
segments. Is `2500` imposed by the ECG-FM checkpoint, or is it our own preprocessing
choice?

**Short answer.** It is *our* choice. `2500` comes from
`SegmentNonoverlapping(segment_length=N_SAMPLES)` in `build_schema_and_transforms()`,
where `N_SAMPLES = SAMPLE_RATE * SEGMENT_SECONDS = 500 * 5 = 2500`. That mirrors
ECG-FM's own inference convention (5 s segments, then aggregate predictions per
record; see `ECG-FM/notebooks/infer_quickstart.ipynb`).

The checkpoint config imposes **no** fixed input length:

- `data/ckpts/mimic_iv_ecg_finetuned.yaml` sets `task._name: ecg_classification` with
  `enable_padding: true` and **does not set** `max_sample_size`.
- `fairseq_signals/tasks/ecg_pretraining.py` declares `max_sample_size: Optional[int] =
  field(default=None)` ("max sample size to crop to for batching") — so with it unset,
  nothing is cropped.
- The architecture is wav2vec2-style: a conv feature encoder
  (`conv_feature_layers = "[(256, 2, 2)] * 4"` → 4 strides of 2 → **16×** subsampling)
  followed by a transformer with a *convolutional* positional encoding
  (`ConvPositionalEncoding`), not learned fixed-length position embeddings. Convolutional
  positions handle arbitrary sequence lengths.

So the encoder accepts variable-length input. This notebook feeds it a full **5000**-sample
PTB-XL record (10 s @ 500 Hz — the raw record length, before segmentation) and checks the
output shape.

**Prediction.** 16× subsampling → `5000 / 16 = 312.5 → 312` tokens, i.e. exactly 2× the
`156` tokens we get from a 2500-sample segment.

In [1]:
import os
import sys

import numpy as np
import torch
import wfdb

# Run from repo root; import the project's own data/encoder helpers.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "notebooks":
    REPO_ROOT = os.getcwd()
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from ecg_transform.inp import ECGInput, ECGInputSchema
from ecg_transform.sample import ECGMetadata, ECGSample
from ecg_transform.t.common import HandleConstantLeads, LinearResample, ReorderLeads
from ecg_transform.t.scale import Standardize

from data import ECG_FM_LEAD_ORDER, SAMPLE_RATE, fix_leads
from encoder import download_checkpoint, encode, load_encoder

DEVICE = "cpu"
PTB_ROOT = os.path.join(REPO_ROOT, "data/ptbxl")
CKPT_DIR = os.path.join(REPO_ROOT, "data/ckpts")

## 1. Load one raw PTB-XL record (5000 samples, no segmentation)

In [2]:
# A records500 PTB-XL file is 10 s @ 500 Hz = 5000 samples.
record_path = os.path.join(PTB_ROOT, "records500/00000/00020_hr")
sig, meta = wfdb.rdsamp(record_path)
print("raw signal shape (samples, leads):", sig.shape)
print("sample rate (Hz):", meta["fs"])
print("duration (s):", sig.shape[0] / meta["fs"])

feats = sig.T.astype(np.float32)  # (leads, samples) = (12, 5000)
print("feats shape (leads, samples):", feats.shape)

raw signal shape (samples, leads): (5000, 12)
sample rate (Hz): 500
duration (s): 10.0
feats shape (leads, samples): (12, 5000)


## 2. Apply ECG-FM transforms *without* `SegmentNonoverlapping`

This is the only change from `src/data.py::build_schema_and_transforms()`: we drop the
segmentation step and set `required_num_samples` to the record's full length, so the whole
5000-sample record reaches the encoder as a single input. Everything else (lead reorder,
resample, constant-lead handling, standardize) is identical to the production pipeline.

In [3]:
n_samples_full = feats.shape[1]  # 5000

schema = ECGInputSchema(
    sample_rate=SAMPLE_RATE,
    expected_lead_order=ECG_FM_LEAD_ORDER,
    required_num_samples=n_samples_full,
)
transforms = [
    ReorderLeads(expected_order=ECG_FM_LEAD_ORDER, missing_lead_strategy="raise"),
    LinearResample(desired_sample_rate=SAMPLE_RATE),
    HandleConstantLeads(strategy="zero"),
    Standardize(),
    # NOTE: no SegmentNonoverlapping -> keep the full 5000-sample record
]

md = ECGMetadata(
    sample_rate=meta["fs"],
    num_samples=feats.shape[1],
    lead_names=fix_leads(meta["sig_name"]),
    unit=None,
    input_start=0,
    input_end=feats.shape[1],
)
md.file = record_path

inp = ECGInput(feats, md)
sample = ECGSample(inp, schema, transforms)

source = torch.from_numpy(sample.out).float()
if source.dim() == 2:
    source = source.unsqueeze(0)  # (12, 5000) -> (1, 12, 5000)
print("model input shape (batch, leads, samples):", tuple(source.shape))

model input shape (batch, leads, samples): (1, 12, 5000)


## 3. Run the encoder

In [4]:
ckpt_path = download_checkpoint(CKPT_DIR)
model = load_encoder(ckpt_path, device=DEVICE)

out = encode(model, source, device=DEVICE)
encoder_out = out["encoder_out"]
logits = out["logits"]

print("encoder_out shape (batch, tokens, dim):", tuple(encoder_out.shape))
print("logits shape      (batch, labels)    :", tuple(logits.shape))

/Users/louissalanon/miniforge3/envs/ecgfm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


encoder_out shape (batch, tokens, dim): (1, 312, 768)
logits shape      (batch, labels)    : (1, 17)


## 4. Check the subsampling arithmetic against the 2500-sample case

In [5]:
n_tokens = encoder_out.shape[1]
print(f"input samples : {source.shape[-1]}")
print(f"output tokens : {n_tokens}")
print(f"effective subsampling : {source.shape[-1] / n_tokens:.2f}x  (expected ~16x)")
print(f"tokens for 2500-sample segment would be ~{2500 // 16} (README reports 156)")
print(f"ratio 5000-token / 2500-token count : {n_tokens / 156:.2f}x  (expected ~2x)")

input samples : 5000
output tokens : 312
effective subsampling : 16.03x  (expected ~16x)
tokens for 2500-sample segment would be ~156 (README reports 156)
ratio 5000-token / 2500-token count : 2.00x  (expected ~2x)


## Conclusion

The encoder runs on a full 5000-sample record with no error and returns
`encoder_out` of shape `(1, 312, 768)` and `logits` of shape `(1, 17)`. The token count
scales linearly with input length (312 ≈ 5000 / 16, exactly 2× the 156 tokens from a
2500-sample segment), confirming the 16× conv subsampling and the convolutional
positional encoding's length-agnostic behaviour.

**Therefore `2500` is a preprocessing choice, not a model constraint.** It comes from
ECG-FM's 5 s-segment inference convention (segment, then aggregate per record), not from
any fixed input length in the checkpoint. Feeding whole 10 s records is architecturally
valid; whether it is *appropriate* is a separate question — the finetuned classification
head and its per-label aggregation (`AGG_METHODS` in ECG-FM's quickstart) were tuned on 5 s
segments, so downstream label calibration may differ for full-record inputs even though the
`encoder_out` representations themselves are well-defined.